## Imports and Settings 

In [2]:
import os         # to check/remove the db file
import sqlite3    # the database (ships with Python, no install needed)
import pandas as pd

SRC = "HI-Small_Trans.csv"   # point this at your downloaded file
OUT = "aml.db"               # the database we're building
CLEAN_SAMPLE = 250_000       # how many non-flagged rows to keep
SEED = 42                    # fixes the random sample so it's reproducible

## Load the flat CSV and clean the column names

In [3]:

df = pd.read_csv(SRC)

# The source file has TWO columns both literally named "Account"
# (one for the sender, one for the receiver). We rename every column
# explicitly to clean snake_case so they're unambiguous from here on.
df.columns = [
    "timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid",
    "payment_currency", "payment_format", "is_laundering",
]

df.head()   # peek at the raw flat table — one row per transaction

,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


## Shrink the data for a fast demo

In [4]:
# Keep EVERY flagged (laundering) row — that's the interesting signal.
flagged = df[df.is_laundering == 1]

# Randomly sample 250k of the clean rows so the db loads/queries instantly.
# random_state=SEED means you get the same sample every time you run it.
clean = df[df.is_laundering == 0].sample(n=CLEAN_SAMPLE, random_state=SEED)

# Stack flagged + clean back together, sort by time, reset the row index.
work = pd.concat([flagged, clean]).sort_values("timestamp").reset_index(drop=True)

# Convert the timestamp text into real datetimes so date questions work later.
work["timestamp"] = pd.to_datetime(work["timestamp"], format="%Y/%m/%d %H:%M")

# Checkpoint: expect ~255k rows and ~2% flagged (we oversampled flags on purpose).
print(len(work), "rows |", round(100 * work.is_laundering.mean(), 2), "% flagged")
work.head()

255177 rows | 2.03 % flagged


,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022-09-01,252389,813BC5980,1547,804B734B0,148.01,Euro,148.01,Euro,Credit Card,0
1,2022-09-01,213,80B745FD0,131086,80BD44380,3534.50,Mexican Peso,3534.50,Mexican Peso,Cheque,0
2,2022-09-01,17400,8067481D0,17400,8067481D0,20.33,US Dollar,20.33,US Dollar,Reinvestment,0
3,2022-09-01,20,80317B030,20,80317B030,150.66,Euro,150.66,Euro,Reinvestment,0
4,2022-09-01,21550,811C7D740,21550,811C7D740,765945.44,US Dollar,765945.44,US Dollar,Reinvestment,0


## Build the "accounts" dimension (the stack-and-dedupe trick)

In [5]:
# Grab the sender side and rename to generic column names.
f = work[["from_bank", "from_account"]].rename(
    columns={"from_bank": "bank", "from_account": "account_id"})

# Grab the receiver side and rename to the SAME generic names.
t = work[["to_bank", "to_account"]].rename(
    columns={"to_bank": "bank", "to_account": "account_id"})

# Stack both halves into one long list of every account-appearance,
# then collapse to one row per unique account.
# (~8 accounts appear under two banks; keeping the first is fine here.)
accounts = pd.concat([f, t]).drop_duplicates("account_id").reset_index(drop=True)

print(len(accounts), "unique accounts")
accounts.head()

203060 unique accounts


,bank,account_id
0,252389,813BC5980
1,213,80B745FD0
2,17400,8067481D0
3,20,80317B030
4,21550,811C7D740


## Write both tables into the database and index them

In [6]:
# Start fresh each run so re-running doesn't pile up duplicate data.
if os.path.exists(OUT):
    os.remove(OUT)

con = sqlite3.connect(OUT)   # creates the aml.db file

# Pour each DataFrame into a table: accounts -> "accounts", work -> "transactions".
accounts.to_sql("accounts", con, index=False)
work.to_sql("transactions", con, index=False)

# Indexes = the textbook index at the back. They make lookups on these
# columns instant instead of scanning all 255k rows. We index exactly the
# columns the agent will join and filter on: sender, receiver, and the flag.
cur = con.cursor()
cur.execute("CREATE INDEX idx_from ON transactions(from_account)")
cur.execute("CREATE INDEX idx_to   ON transactions(to_account)")
cur.execute("CREATE INDEX idx_flag ON transactions(is_laundering)")
con.commit()   # save the changes

print("DB written:", OUT, round(os.path.getsize(OUT) / 1e6, 1), "MB")

DB written: aml.db 40.9 MB


## Joining the tables

In [7]:
import eval
eval.main()

      type                                      question  ran_sql result
   factual How many transactions were flagged as launder     True   PASS
   factual     How many transactions are there in total?     True   PASS
   factual How many unique accounts are in the database?     True   PASS
   factual   How many distinct payment formats are used?     True   PASS
   factual            How many distinct banks are there?     True   PASS
   factual  How many distinct payment currencies appear?     True   PASS
   factual      Which payment format is used most often?     True   PASS
   factual How many transactions used the Bitcoin paymen     True   PASS
   factual How many transactions used the Cheque payment     True   PASS
   factual How many transactions had an amount paid over     True   PASS
   factual How many transactions were self-transfers (sa     True   PASS
   factual How many flagged transactions had the same se     True   PASS
   factual How many transactions used a different r

In [8]:
import agent, sqlite3, pandas as pd
r = agent.run("What are the top 5 banks by total flagged money received?")
print("AGENT GOT:\n", r.final_result, "\n")
con = sqlite3.connect("aml.db")
print("REFERENCE:\n", pd.read_sql(
    "SELECT to_bank, ROUND(SUM(amount_received)) FROM transactions "
    "WHERE is_laundering=1 GROUP BY to_bank ORDER BY SUM(amount_received) DESC LIMIT 5", con))
con.close()

AGENT GOT:
    to_bank  total_flagged_received
0   116781            8.485572e+10
1    13029            6.641642e+10
2      213            2.150094e+10
3   148016            2.627196e+09
4    42596            2.486913e+09 

REFERENCE:
    to_bank  ROUND(SUM(amount_received))
0   116781                 8.485572e+10
1    13029                 6.641642e+10
2      213                 2.150094e+10
3   148016                 2.627196e+09
4    42596                 2.486913e+09
